In [5]:
import pandas as pd

# Put an 'r' before the string to prevent errors caused by backslashes
df = pd.read_csv("../data/raw/results.csv")
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49385,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49386,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49387,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49388,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


In [6]:
df['date'] = pd.to_datetime(df['date'])

In [7]:
# 2. Filter for games from 2014 onward
recent_games = df[df['date'] >= '2014-01-01']

# View the first few rows of your new filtered dataset
recent_games.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
37558,2014-01-01,Kuwait,Jordan,1.0,2.0,WAFF Championship,Doha,Qatar,True
37559,2014-01-04,Bahrain,Jordan,0.0,1.0,WAFF Championship,Doha,Qatar,True
37560,2014-01-04,Namibia,Ghana,0.0,1.0,Friendly,Windhoek,Namibia,False
37561,2014-01-04,Nigeria,Ethiopia,2.0,1.0,Friendly,Abuja,Nigeria,False
37562,2014-01-04,Qatar,Kuwait,3.0,0.0,WAFF Championship,Doha,Qatar,False


In [8]:
# 1. Define your extensive list of teams
my_teams = [
    "Canada", "Mexico", "United States", "Australia", "Iraq", "IR Iran", 
    "Japan", "Jordan", "Korea Republic", "South Korea", "Qatar", "Saudi Arabia", 
    "Uzbekistan", "Algeria", "Cabo Verde", "DR Congo", "Côte d'Ivoire", "Egypt", 
    "Ghana", "Morocco", "Senegal", "South Africa", "Tunisia", "Curaçao", 
    "Haiti", "Panama", "Argentina", "Brazil", "Colombia", "Ecuador", 
    "Paraguay", "Uruguay", "New Zealand", "Austria", "Belgium", "Bosnia and Herzegovina", 
    "Croatia", "Czechia", "England", "France", "Germany", "Netherlands", 
    "Norway", "Portugal", "Scotland", "Spain", "Sweden", "Switzerland", "Türkiye"
]

# 2. Filter where BOTH teams are in your list (from 2014-present)
clash_df = recent_games[
    recent_games['home_team'].isin(my_teams) & 
    recent_games['away_team'].isin(my_teams)
]

# View the total number of matches found and the data
print(f"Found {len(clash_df)} matches between these teams since 2014.")
clash_df.head()

Found 1326 matches between these teams since 2014.


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
37564,2014-01-07,Qatar,Jordan,2.0,0.0,WAFF Championship,Doha,Qatar,False
37574,2014-01-29,Mexico,South Korea,4.0,0.0,Friendly,San Antonio,United States,True
37576,2014-02-01,United States,South Korea,2.0,0.0,Friendly,Carson,United States,False
37587,2014-03-05,Australia,Ecuador,3.0,4.0,Friendly,London,England,True
37588,2014-03-05,Austria,Uruguay,1.0,1.0,Friendly,Klagenfurt,Austria,False


In [9]:
print(df['tournament'].value_counts().head(20))

tournament
Friendly                                18312
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


In [10]:
print(df[df['tournament'].str.contains('World Cup', na=False)]['tournament'].unique())

<StringArray>
[                'FIFA World Cup',   'FIFA World Cup qualification',
                 'Viva World Cup', 'CONIFA World Cup qualification']
Length: 4, dtype: str


In [11]:
# Teams in your manual list
my_teams_set = set(my_teams)

# Teams actually appearing in the filtered data
data_teams_set = set(clash_df['home_team'].unique()) | set(clash_df['away_team'].unique())

# Check for mismatches
print("Teams in manual list but NOT in data:")
print(sorted(my_teams_set - data_teams_set))

print("\nTeams in data but NOT in manual list:")
print(sorted(data_teams_set - my_teams_set))

Teams in manual list but NOT in data:
['Cabo Verde', 'Czechia', "Côte d'Ivoire", 'IR Iran', 'Korea Republic', 'Türkiye']

Teams in data but NOT in manual list:
[]


In [12]:
# For each team in your manual list that didn't match, find what the data calls them
problem_teams = ['Cabo Verde', 'Czechia', "Côte d'Ivoire", 'IR Iran', 'Korea Republic', 'Türkiye']

for team in problem_teams:
    # Try to find matches in the full dataset (before filtering)
    home_matches = df[df['home_team'].str.contains(team[:4], case=False, na=False)]['home_team'].unique()
    away_matches = df[df['away_team'].str.contains(team[:4], case=False, na=False)]['away_team'].unique()
    
    all_variants = set(home_matches) | set(away_matches)
    print(f"{team}: {all_variants}")

Cabo Verde: set()
Czechia: {'Czech Republic', 'Czechoslovakia'}
Côte d'Ivoire: set()
IR Iran: set()
Korea Republic: {'South Korea', 'United Koreans in Japan', 'North Korea'}
Türkiye: set()


In [13]:
# Check for the most likely actual names in the dataset
likely_names = {
    'Cabo Verde': ['Cape Verde'],
    'Czechia': ['Czech Republic'],
    "Côte d'Ivoire": ['Ivory Coast'],
    'IR Iran': ['Iran'],
    'Korea Republic': ['South Korea'],
    'Türkiye': ['Turkey']
}

for canonical, variants in likely_names.items():
    for variant in variants:
        count_home = len(df[df['home_team'] == variant])
        count_away = len(df[df['away_team'] == variant])
        if count_home > 0 or count_away > 0:
            print(f"'{canonical}' → '{variant}': {count_home + count_away} matches")

'Cabo Verde' → 'Cape Verde': 237 matches
'Czechia' → 'Czech Republic': 363 matches
'Côte d'Ivoire' → 'Ivory Coast': 639 matches
'IR Iran' → 'Iran': 615 matches
'Korea Republic' → 'South Korea': 1010 matches
'Türkiye' → 'Turkey': 644 matches


In [14]:
TEAM_NAME_MAP = {
    'Cape Verde': 'Cabo Verde',
    'Czech Republic': 'Czechia',
    'Ivory Coast': "Côte d'Ivoire",
    'Iran': 'IR Iran',
    'South Korea': 'Korea Republic',
    'Turkey': 'Türkiye'
}

# Apply the mapping to both home_team and away_team columns
clash_df['home_team'] = clash_df['home_team'].replace(TEAM_NAME_MAP)
clash_df['away_team'] = clash_df['away_team'].replace(TEAM_NAME_MAP)

# Verify the mapping worked
print(f"Unique teams after mapping: {len(clash_df['home_team'].unique())}")
print(f"Total matches: {len(clash_df)}")

# Check that all teams in data now match your manual list
data_teams = set(clash_df['home_team'].unique()) | set(clash_df['away_team'].unique())
print(f"\nTeams in data now: {len(data_teams)}")
print(f"Match count: {data_teams == set(my_teams)}")

Unique teams after mapping: 44
Total matches: 1326

Teams in data now: 44
Match count: False


In [15]:
data_teams = set(clash_df['home_team'].unique()) | set(clash_df['away_team'].unique())
manual_teams = set(my_teams)

print("Teams in manual list but NOT in filtered data:")
missing = sorted(manual_teams - data_teams)
print(missing)
print(f"\nCount: {len(missing)}")

Teams in manual list but NOT in filtered data:
['Cabo Verde', 'Czechia', "Côte d'Ivoire", 'IR Iran', 'South Korea', 'Turkey']

Count: 6


In [16]:
# Check what team names are actually in the data after replace
print("Sample of home teams after mapping:")
print(clash_df['home_team'].unique()[:10])

# Specifically check for the mapped names
print("\n'Czechia' in data:", 'Czechia' in clash_df['home_team'].values)
print("'Czech Republic' in data:", 'Czech Republic' in clash_df['home_team'].values)
print("'Korea Republic' in data:", 'Korea Republic' in clash_df['home_team'].values)
print("'South Korea' in data:", 'South Korea' in clash_df['home_team'].values)

Sample of home teams after mapping:
<StringArray>
[                 'Qatar',                 'Mexico',          'United States',
              'Australia',                'Austria', 'Bosnia and Herzegovina',
               'Colombia',                 'France',                  'Japan',
           'South Africa']
Length: 10, dtype: str

'Czechia' in data: False
'Czech Republic' in data: False
'Korea Republic' in data: True
'South Korea' in data: False


In [17]:
# Start fresh from the original unfiltered dataframe
df_clean = df[df['date'] >= '2014-01-01'].copy()

# Apply the mapping FIRST
TEAM_NAME_MAP = {
    'Cape Verde': 'Cabo Verde',
    'Czech Republic': 'Czechia',
    'Ivory Coast': "Côte d'Ivoire",
    'Iran': 'IR Iran',
    'South Korea': 'Korea Republic',
    'Turkey': 'Türkiye'
}

df_clean['home_team'] = df_clean['home_team'].replace(TEAM_NAME_MAP)
df_clean['away_team'] = df_clean['away_team'].replace(TEAM_NAME_MAP)

# THEN filter to your qualified teams (with the corrected manual list)
my_teams = [
    'Canada', 'Mexico', 'United States', 'Australia', 'Iraq', 'IR Iran',
    'Japan', 'Jordan', 'Korea Republic', 'Qatar', 'Saudi Arabia',
    'Uzbekistan', 'Algeria', 'Cabo Verde', 'DR Congo', "Côte d'Ivoire", 'Egypt',
    'Ghana', 'Morocco', 'Senegal', 'South Africa', 'Tunisia', 'Curaçao',
    'Haiti', 'Panama', 'Argentina', 'Brazil', 'Colombia', 'Ecuador',
    'Paraguay', 'Uruguay', 'New Zealand', 'Austria', 'Belgium',
    'Bosnia and Herzegovina', 'Croatia', 'Czechia', 'England', 'France',
    'Germany', 'Netherlands', 'Norway', 'Portugal', 'Scotland', 'Spain',
    'Sweden', 'Switzerland', 'Türkiye'  # Removed duplicate 'Turkey'
]

clash_df = df_clean[
    (df_clean['home_team'].isin(my_teams)) &
    (df_clean['away_team'].isin(my_teams))
]

print(f"Unique teams: {len(clash_df['home_team'].unique() | set(clash_df['away_team'].unique()))}")
print(f"Total matches: {len(clash_df)}")

TypeError: unsupported operand type(s) for |: 'StringArray' and 'set'

In [18]:
unique_teams = set(clash_df['home_team'].unique()) | set(clash_df['away_team'].unique())
print(f"Unique teams: {len(unique_teams)}")
print(f"Total matches: {len(clash_df)}")

Unique teams: 48
Total matches: 1505


In [20]:
# Save the filtered dataset
clash_df.to_csv('../data/processed/matches_filtered.csv', index=False)
print("Saved to data/processed/matches_filtered.csv")

# Save the qualified teams list
qualified_teams_df = pd.DataFrame({'team': sorted(my_teams)})
qualified_teams_df.to_csv('../data/processed/wc2026_teams.csv', index=False)
print("Saved to data/processed/wc2026_teams.csv")

# Final sanity check
print(f"\nPhase 1 complete:")
print(f"  Matches: {len(clash_df)}")
print(f"  Teams: {len(my_teams)}")
print(f"  Date range: {clash_df['date'].min()} to {clash_df['date'].max()}")
print(f"  Tournament types: {clash_df['tournament'].nunique()} unique")

Saved to data/processed/matches_filtered.csv
Saved to data/processed/wc2026_teams.csv

Phase 1 complete:
  Matches: 1505
  Teams: 48
  Date range: 2014-01-07 00:00:00 to 2026-06-27 00:00:00
  Tournament types: 26 unique
